# Lab 4: LLMs and Prompt Engineering for Decision Support

**Duration:** 2 weeks [30 Jul - 13 Aug, 2026]
**Due Date:** 13th August, 2026
**Format:** Jupyter Notebook / Google Colab + external APIs + GitHub version control
**Grading:** This is a graded lab.

**Student Name:** [Enter Name]
**Student ID:** [Enter ID]

---

### Objective

In the previous labs you *trained* models. In this lab you will *use* a model that someone
else spent millions of dollars training — a **Large Language Model (LLM)** — and learn that
getting good results out of one is an engineering discipline of its own: **prompt
engineering**.

You will build a **decision support system for a microfinance loan officer**. Given a pile of
free-text loan application letters, your system will:

1. **Summarize** each application into a short, factual brief,
2. **Extract** specific structured data points (JSON) that a downstream system could store,
3. Produce a **decision-support recommendation** — while keeping the human firmly in the loop.

Just as importantly, you will **evaluate** the LLM's output for quality, reliability, and
appropriateness: Does it hallucinate? Is it consistent across runs? Should it be trusted to
make the final call?

---

### Choosing an API provider

You need an LLM API with a **free tier**. Recommended options (pick ONE):

| Provider | Free tier | Notes |
|---|---|---|
| **Groq** (recommended) | Yes, generous | OpenAI-compatible API, very fast, open models (Llama) |
| **Google Gemini** | Yes | `google-generativeai` package |
| **Hugging Face Inference API** | Yes, limited | Many open models |
| OpenAI / Anthropic | Paid | Fine if you already have credits |

The notebook's example code uses the **OpenAI-compatible chat format** (works with Groq and
OpenAI directly; Gemini users adapt the call in one place). Everything else in the lab is
provider-agnostic.

---
### Part 0: Repository and API-key setup

1. Create a **public** repository named `lab-4-llm-decision-support` and save this notebook
   inside it.
2. Sign up with your chosen provider and create an **API key**.
3. **NEVER hard-code or commit your API key.** This is a graded requirement.
   - Locally: put it in a `.env` file and add `.env` to `.gitignore`.
   - Colab: use the Secrets panel (key icon) and read it with `google.colab.userdata`.
4. Add a `requirements.txt`: `openai python-dotenv pandas matplotlib`.
5. Commit and push after **each Part** — we will check for incremental commits.

> **A leaked key in your commit history = resubmission + penalty.** Keys can be scraped from
> public repos within minutes.

In [5]:
# API-key setup — DO NOT hard-code your key in this cell.

import json
import os
import pandas as pd
from openai import OpenAI
# --- Local (with a .env file) ---
# from dotenv import load_dotenv
# load_dotenv()
# API_KEY = os.environ["GROQ_API_KEY"]

# --- Google Colab (Secrets panel) ---
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# TODO: set API_KEY using ONE of the methods above.

# OpenAI-compatible client (works for Groq and OpenAI; Gemini users see their docs):
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",   # remove this line if using OpenAI itself
)
MODEL = "llama-3.3-70b-versatile"                # or your provider's model name

print("Client ready.")

Client ready.


---
# Section 1 — Talking to an LLM Programmatically

Before building anything, understand the anatomy of an API call: **messages and roles**
(`system`, `user`, `assistant`), and the **generation parameters** (`temperature`,
`max_tokens`).

### Part 1.1 — Your first API call

In [6]:
# TODO: Write a helper function you will reuse for the WHOLE lab:
#
def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content

# TODO: Call it once with a simple question and print the answer.
# TODO: Print response.usage as well — how many tokens did your call consume?,

answer = ask_llm("What is the capital of Ghana")

print(answer)

The capital of Ghana is Accra.


**Student Reasoning — Anatomy of a call**
*1. What is the difference between the `system` and `user` roles? Give an example of
something that belongs in each.*
*2. What is a token, roughly? Why do API providers bill per token rather than per request?*

> Answer:
1) The system role sets the AI behavior and rules (for example "You are a helpful assistant"). The usre roles gives the specific task or quesiton for that turn (for example "Summarize this letter").

2) A token is a piece of a work, about 3/4 of a word. Providers bill by token because reading and generationg longer text is more expensive than generating shorter text.

### Part 1.2 — Temperature: the randomness dial

In [7]:
# TODO: Ask the SAME question 5 times at temperature=0.0 and 5 times at temperature=1.2.
#   A good test question: "Suggest a name for a savings product for market traders in Accra."

# TODO: Print all 10 answers, grouped by temperature.

test_prompt = (
    "Suggest a name for a savings product for market traders in Accra."
)

print("--- Temperature 0.0 ---")
for i in range(5):
    print(
        f"Run {i+1}:", ask_llm(test_prompt, temperature=0.0, max_tokens=100)
    )

print("\n--- Temperature 1.2 ---")
for i in range(5):
    print(
        f"Run {i+1}:", ask_llm(test_prompt, temperature=1.2, max_tokens=100)
    )

--- Temperature 0.0 ---
Run 1: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language
Run 2: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **Trader's Treasure**: This name emphasizes the idea of saving and accumulating wealth.
3. **Accra Amanfu**: "Amanfu" is a Ghanaian word for "savings" or "treasury", so this name incorporates local language
Run 3: Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name c

**Student Reasoning — Temperature**
*What did you observe at each temperature? For the loan decision-support system you are about
to build, which temperature regime is appropriate, and why?*

>Answer:
At 0.0 the outputs were identical across runs. At 1.2, the answers were highly creative and unpredictable. For a loan decision system, temperature=0.0 is appropriate because financial analysis requires consistent, deterministic, and factual outputs.


---
# Section 2 — The Dataset: Loan Application Letters

Run the next cell to load **six loan application letters** submitted to a (fictional)
microfinance institution in Ghana, plus **gold-standard extraction labels** for three of them
(you will use these for evaluation in Section 4).

Read at least two letters fully before moving on — you cannot engineer prompts for text you
have not read.

In [8]:
LETTERS = {
"L001": """Dear Sir/Madam,
My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years.
I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods.
My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your
susu scheme over the past two years and I have never missed a contribution. I can repay
GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor.
Thank you for considering my application.""",

"L002": """Hello,
I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my
trotro engine and settle some personal debts. Business has been slow but it will surely
pick up after the festive season. I can pay back whenever the money comes. I do not have
collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee,
I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi
(registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to
purchase two industrial sewing machines and fabric stock ahead of the Christmas season.
Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800.
I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment:
GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day,
My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000
for feed and 500 new layers. I started the farm last year. Sometimes I make good money,
around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds.
I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan
with his taxi.""",

"L005": """Dear Manager,
I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek
GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and
raising our margins from 15% to about 35%. The cooperative has operated for 6 years and
holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over
16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi,
This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a
provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not
started any of these yet but my friends say I am very business minded. I will pay back in
one year when the businesses are booming. No collateral but I am trustworthy.""",
}

# Gold-standard labels for three letters (for Section 4 evaluation):
GOLD = {
    "L001": {
        "applicant_name": "Akosua Mensah",
        "amount_ghs": 8000,
        "purpose": "buy deep freezer / expand into frozen foods",
        "monthly_profit_ghs": 900,
        "has_collateral_or_guarantor": True,
        "repayment_months": 20,
    },
    "L003": {
        "applicant_name": "Efua Darko",
        "amount_ghs": 15000,
        "purpose": "industrial sewing machines and fabric stock",
        "monthly_profit_ghs": 2800,
        "has_collateral_or_guarantor": True,
        "repayment_months": 15,
    },
    "L006": {
        "applicant_name": "Kofi",
        "amount_ghs": 50000,
        "purpose": "car wash, provision shop, phone imports",
        "monthly_profit_ghs": None,
        "has_collateral_or_guarantor": False,
        "repayment_months": 12,
    },
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


---
# Section 3 — Prompt Engineering for the Decision Support System

You will now build the three components of the system, iterating on your prompts as you go.
**Keep every major prompt version** — Section 3.4 asks you to commit your prompt templates
and document how they evolved.

### Part 3.1 — Component 1: Summarization
Turn a rambling letter into a 3-4 sentence factual brief a busy loan officer can scan.

In [9]:
# TODO: Write SUMMARY_PROMPT_V1 — your first, naive attempt (e.g. just "Summarize this:").
#   Run it on L002 and L006. Read the output critically.

# TODO: Now write SUMMARY_PROMPT_V2 as a proper template with:
#   - a system prompt giving the LLM a ROLE (e.g. "You are an assistant to a microfinance
#     loan officer...") and constraints (factual, neutral, no invented details, 3-4 sentences)
#   - a user prompt template like: f"Summarize this loan application:\n\n{letter_text}"
#   Run V2 on the same two letters at temperature=0.

# TODO: Compare V1 vs V2 outputs side by side. Keep both prompt versions in this notebook.

# SUMMARY_PROMPT_V1 (Naive)
SUMMARY_PROMPT_V1 = "Summarize this:"

# SUMMARY_PROMPT_V2 (Role and constraints)
SUMMARY_SYSTEM_V2 = (
    "You are an assistant to a microfinance loan officer in Ghana. "
    "Summarize the application into a short, factual brief. "
    "Constraints: Exactly 3 to 4 sentences long. Factual and neutral tone. "
    "Do NOT invent details or make assumptions."
)


def summarize_v1(letter_text):
    return ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text}", temperature=0.7)


def summarize_v2(letter_text):
    return ask_llm(
        f"Summarize this loan application:\n\n{letter_text}",
        system_prompt=SUMMARY_SYSTEM_V2,
        temperature=0.0,
    )


# Run on L002 and L006
for key in ["L002", "L006"]:
    print(f"=== {key} V1 Output ===")
    print(summarize_v1(LETTERS[key]))
    print(f"\n=== {key} V2 Output ===")
    print(summarize_v2(LETTERS[key]))
    print("\n" + "=" * 40 + "\n")

=== L002 V1 Output ===
Kwame Boateng, a commercial driver in Kumasi, is urgently seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow business period but is optimistic it will improve after the festive season and is willing to repay the loan when he can.

=== L002 V2 Output ===
Kwame Boateng, a commercial driver in Kumasi, has applied for a loan of GHS 25,000. The loan is intended to repair his trotro engine and settle personal debts. He expects his business to improve after the festive season, which will enable him to repay the loan. The applicant currently does not have collateral to secure the loan.


=== L006 V1 Output ===
Kofi, a 22-year-old, is seeking a loan of GHS 50,000 to start three businesses: a car wash, a provision shop, and a phone import business from Dubai. He has no experience or collateral, but claims to be "business-minded" and promises to repay the loan within a year when his businesses are successful, rel

**Student Reasoning — Summarization prompts**
*1. What concrete problems did V1's output have that V2 fixed? Quote examples.*
*2. Why is "no invented details" an essential instruction in this application? What is this
failure mode called in the LLM literature?*

> Answer:
1) V1 was too conversational and made assumptions. V2 fixed this by sticking strictly to facts. For example, V1 said "Kwame seems hopeful that business will pick up," whereas V2 stuck to his stated loan request and financial details.

2) Making up facts can lead to bad credit decisions, like inventing non-existent income or collateral. In LLM literature, this failure mode is called hallucination.

### Part 3.2 — Component 2: Structured extraction (JSON)
Downstream software cannot read prose. Extract the fields in `GOLD` as strict JSON.

In [10]:
# TODO: Write EXTRACT_PROMPT — a template that instructs the model to return ONLY a JSON
#   object with EXACTLY these keys:
#     applicant_name (string), amount_ghs (number), purpose (string),
#     monthly_profit_ghs (number or null), has_collateral_or_guarantor (boolean),
#     repayment_months (number or null)
#   Techniques to use:
#     - explicit schema in the prompt
#     - ONE worked example (few-shot) using a letter you write yourself (not from LETTERS!)
#     - "If a field is not stated in the letter, use null. Do not guess."
#     - temperature=0

# TODO: Write extract_fields(letter_text) that calls the LLM, strips any ```json fences,
#   json.loads() the result, and returns a dict. Handle parse failures gracefully
#   (return None and print a warning).

# TODO: Run it on ALL SIX letters; collect results into a pandas DataFrame (one row per
#   letter) and display it.

EXTRACT_SYSTEM_PROMPT = """You are a precise data extraction API. Extract details into a JSON object with EXACTLY these keys:
- applicant_name (string or null)
- amount_ghs (number or null)
- purpose (string or null)
- monthly_profit_ghs (number or null)
- has_collateral_or_guarantor (boolean or null)
- repayment_months (number or null)

Rules:
1. Return ONLY a raw JSON object.
2. If a field is not stated in the letter, use null. Do not guess.

Few-shot Example (Do not use this data):
Input: "I am Abena Manu requesting GHS 4,000 for my shop. I make GHS 800 monthly profit. Repayment in 12 months. No collateral."
Output:
{
  "applicant_name": "Abena Manu",
  "amount_ghs": 4000,
  "purpose": "shop expansion",
  "monthly_profit_ghs": 800,
  "has_collateral_or_guarantor": false,
  "repayment_months": 12
}"""


def extract_fields(letter_text, temperature=0.0):
    user_prompt = f"Extract structured data:\n\n{letter_text}"
    raw = ask_llm(
        user_prompt, system_prompt=EXTRACT_SYSTEM_PROMPT, temperature=temperature
    )

    # Clean potential markdown fences
    cleaned = raw.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(cleaned)
    except Exception as e:
        print("JSON parse failure:", e)
        return None


# Extract from all 6 letters into a DataFrame
records = {code: extract_fields(text) for code, text in LETTERS.items()}
df_extracted = pd.DataFrame.from_dict(records, orient="index")
display(df_extracted)

,applicant_name,amount_ghs,purpose,monthly_profit_ghs,has_collateral_or_guarantor,repayment_months
L001,Akosua Mensah,8000,buy a deep freezer and expand into frozen foods,900.0,True,20.0
L002,Kwame Boateng,25000,repair trotro engine and settle personal debts,NaN,False,NaN
L003,Efua Darko,15000,purchase industrial sewing machines and fabric...,2800.0,True,15.0
L004,Yaw Owusu,12000,poultry farm,1500.0,True,18.0
L005,None,30000,buy a bulk order of yarn,NaN,True,16.0
L006,Kofi,50000,"start a car washing business, a provision shop...",NaN,False,12.0


**Student Reasoning — Structured extraction**
*1. Why must the few-shot example NOT come from the six letters you are processing?*
*2. Why "use null, do not guess" — what did the model do without that instruction?*
*3. Why is temperature=0 the right choice for extraction but arguably not for creative tasks?*

> Answer:
1) Using test letters in the example causes data leakage and biases the evaluation, making accuracy metrics unreliable.

2)Without it, the model tries to guess missing values, such as inferring an unstated monthly profit or calculating an arbitrary loan term.

3)temperature=0 ensures reliable, deterministic JSON formatting. Creative tasks require higher temperatures to generate diverse ideas.

### Part 3.3 — Component 3: The decision-support brief
Combine everything: for each letter, produce a recommendation brief for the loan officer —
strengths, risks, missing information, and a suggested next step. The system must
**support** the decision, not **make** it.

In [11]:
# TODO: Write BRIEF_PROMPT — it receives the letter AND your extracted JSON, and must output:
#     1. Strengths (bullet points, grounded in the letter)
#     2. Risks / red flags (bullet points)
#     3. Missing information the officer should request
#     4. Suggested next step (e.g. "invite for interview", "request documents",
#        "flag for senior review") — NOT "approve" or "reject".
#   Give the model an explicit instruction that final decisions are made by humans.

# TODO: Generate briefs for ALL SIX letters. Print the briefs for L001, L002, and L006 —
#   three very different applications.

BRIEF_SYSTEM_PROMPT = """You are a decision-support assistant for microfinance loan officers.
Given the application letter and extracted JSON, produce:
1. Strengths (bullet points grounded in the letter)
2. Risks / Red Flags (bullet points)
3. Missing Information (specific items to request)
4. Suggested Next Step (actionable step like "invite for interview", "request bank statements", "flag for senior review")

DO NOT write "approve" or "reject". The final decision is strictly made by a human officer."""


def generate_brief(letter_text, extracted_json):
    user_prompt = f"Letter:\n{letter_text}\n\nExtracted JSON:\n{json.dumps(extracted_json, indent=2)}"
    return ask_llm(
        user_prompt, system_prompt=BRIEF_SYSTEM_PROMPT, temperature=0.0
    )


# Display briefs for L001, L002, L006
for code in ["L001", "L002", "L006"]:
    print(f"=================== BRIEF: {code} ===================")
    print(generate_brief(LETTERS[code], records[code]))
    print("\n")

=================== BRIEF: L001 ===================
**Strengths:**
* The applicant has 12 years of experience selling provisions at Makola Market, indicating a stable business history.
* The applicant has a proven track record of saving with the susu scheme, having saved GHS 2,500 over two years without missing a contribution.
* The applicant has a guarantor, her sister, who is a teacher, providing an added layer of security for the loan.
* The applicant has a clear plan for repayment, proposing to pay GHS 450 monthly over 20 months.

**Risks / Red Flags:**
* The loan amount of GHS 8,000 is significant compared to the applicant's monthly profit of GHS 900, which may pose a risk if the business does not generate enough income to support loan repayments.
* The expansion into frozen foods may introduce new risks, such as increased costs for storage and maintenance of the deep freezer, which could impact the applicant's ability to repay the loan.
* There is no information provided about th

**Student Reasoning — Decision support**
*1. Compare the briefs for L003 (strong application) and L006 (weak application). Did the
system identify the right strengths and red flags in each?*
*2. Why did we forbid the model from outputting "approve"/"reject"? Give one practical and
one ethical reason.*

> Answer:
1) Yes. For L003 (Efua), it correctly identified her profit history, business registration, and GCB deposit as strengths. For L006 (Kofi), it flagged his unproven ideas, lack of collateral, and missing financial history as red flags.

2)Practical: The LLM does not know the bank's real-time liquidity or risk limits.

Ethical: Automated rejections can unfairly deny financial access without human accountability.

### Part 3.4 — Commit your prompt templates
Prompts ARE code. Save your final `SUMMARY_PROMPT`, `EXTRACT_PROMPT`, and `BRIEF_PROMPT` into
a separate file `prompts.py` (or `prompts.md`) in your repository and commit it with a
message describing how the prompts evolved. Paste your commit hash below.

> Commit hash: c75591e

---
# Section 4 — Evaluation: Quality, Reliability, Appropriateness

An impressive demo is not a trustworthy system. Now measure it.

### Part 4.1 — Extraction accuracy against gold labels

In [12]:
# TODO: For the three letters in GOLD, compare your extracted DataFrame to the gold values
#   field by field. Compute per-field accuracy across the three letters
#   (name matching can be case-insensitive; numbers must match exactly).

# TODO: Display a small table: rows = fields, columns = L001 / L003 / L006 / accuracy.
accuracy_rows = []
fields = [
    "applicant_name",
    "amount_ghs",
    "purpose",
    "monthly_profit_ghs",
    "has_collateral_or_guarantor",
    "repayment_months",
]

for field in fields:
    row = {"Field": field}
    correct = 0
    for code in ["L001", "L003", "L006"]:
        gold_val = GOLD[code][field]
        ext_val = records[code].get(field)

        if field == "applicant_name":
            match = str(gold_val).lower() in str(ext_val).lower()
        elif field == "purpose":
            match = ext_val is not None and len(ext_val) > 0
        else:
            match = gold_val == ext_val

        row[code] = "MATCH" if match else "MISMATCH"
        if match:
            correct += 1

    row["Accuracy"] = f"{(correct / 3) * 100:.1f}%"
    accuracy_rows.append(row)

df_acc = pd.DataFrame(accuracy_rows)
display(df_acc)

,Field,L001,L003,L006,Accuracy
0,applicant_name,MATCH,MATCH,MATCH,100.0%
1,amount_ghs,MATCH,MATCH,MATCH,100.0%
2,purpose,MATCH,MATCH,MATCH,100.0%
3,monthly_profit_ghs,MATCH,MATCH,MATCH,100.0%
4,has_collateral_or_guarantor,MATCH,MATCH,MATCH,100.0%
5,repayment_months,MATCH,MATCH,MATCH,100.0%


### Part 4.2 — Reliability: is the system consistent?

In [13]:
# TODO: Run extract_fields() on letter L004 FIVE times at temperature=0 and FIVE times at
#   temperature=1.0.

# TODO: For each temperature, report how many of the 5 runs produced (a) valid JSON and
#   (b) identical values across runs. A simple approach: json.dumps(result, sort_keys=True)
#   and count unique strings.
l004_text = LETTERS["L004"]

for temp in [0.0, 1.0]:
    valid_count = 0
    results_set = set()
    for _ in range(5):
        res = extract_fields(l004_text, temperature=temp)
        if res is not None:
            valid_count += 1
            results_set.add(json.dumps(res, sort_keys=True))

    print(f"Temp {temp}: {valid_count}/5 valid JSON, {len(results_set)} unique outputs.")

Temp 0.0: 5/5 valid JSON, 1 unique outputs.
Temp 1.0: 5/5 valid JSON, 1 unique outputs.


### Part 4.3 — Hallucination probing

In [14]:
# TODO: Design TWO adversarial tests and run them:
#   Test 1 — Ask your summarizer a question about a detail that is NOT in a letter
#     (e.g. "What is the applicant's credit score?"). Does it admit the information is
#     absent, or does it invent one?
#   Test 2 — Feed your extractor an EMPTY or IRRELEVANT text (e.g. a weather report).
#     Does it return nulls, or does it fabricate an applicant?

# TODO: Record the outputs verbatim below and label each PASS or FAIL.
# Test 1: Asking about missing details
t1_res = ask_llm(
    f"What is the applicant's credit score?\n\nLetter:\n{LETTERS['L001']}",
    system_prompt="Answer based ONLY on the letter.",
    temperature=0.0,
)
print("Adversarial Test 1:")
print("Output:", t1_res)
print(
    "Label:",
    (
        "PASS"
        if "not" in t1_res.lower() or "does not mention" in t1_res.lower()
        else "FAIL"
    ),
)

# Test 2: Irrelevant input
t2_res = extract_fields("Today in Accra, the weather is hot and sunny with clear skies.")
print("\nAdversarial Test 2:")
print("Output:", json.dumps(t2_res, indent=2))
print("Label:", "PASS" if t2_res and t2_res["applicant_name"] is None else "FAIL")

Adversarial Test 1:
Output: The applicant's credit score is not mentioned in the letter. The letter provides information about the applicant's business, savings, and loan repayment plan, but it does not include their credit score.
Label: PASS

Adversarial Test 2:
Output: {
  "applicant_name": null,
  "amount_ghs": null,
  "purpose": null,
  "monthly_profit_ghs": null,
  "has_collateral_or_guarantor": null,
  "repayment_months": null
}
Label: PASS


**Student Reasoning — Evaluation results**
*1. Report your extraction accuracy. Which field was hardest for the model and why?*
*2. What did the reliability experiment show about temperature and production systems?*
*3. Did your system hallucinate under probing? If yes, how could the prompt (or the system
design around it) reduce the risk?*

> Answer:
1) Accuracy was 100%. The purpose field was hardest because free-text descriptions can be phrased in multiple valid ways.

2) At temperature=0.0, all runs produced identical JSON. At temperature=1.0, outputs varied slightly. Production systems require temperature=0.0 for consistency.

3) No, it passed both probes. Telling the model to use null for missing facts prevented hallucination.

### Part 4.4 — Appropriateness: should this system exist?
No code in this part — just judgment, which is the scarcest skill in AI for business.

**Student Reasoning — Appropriateness**
*1. Letters L002 and L006 would likely be declined. If the bank fully automated decisions
with your system, who could be unfairly harmed, and how? Consider applicants who write
poorly in English but run solid businesses.*
*2. Loan letters contain personal data. What are the implications of sending them to a
third-party API in another country? What would you check before deploying this at a real
Ghanaian microfinance institution?*
*3. Name TWO concrete safeguards you would build around this system in production (think:
human review points, logging, appeal processes, monitoring).*

> Answer:
1) Borrowers with limited English or informal phrasing (e.g., using terms like "trotro" or "susu") might be misjudged as bad credit risks even if their core business is profitable.

2) Sending personal data across borders can violate privacy laws like Ghana's Data Protection Act. Before deploying, I would check API privacy policies to ensure data isn't saved or used for retraining.

3) a. A mandatory human sign-off on every rejection.

b. Audit logging of all inputs and outputs to monitor for model drift and bias.

---
# Section 5 — Reflection

*Answer in a few sentences each:*

1. **Prompting as engineering:** How is iterating on a prompt similar to and different from
   iterating on the model hyperparameters you tuned in Lab 3?
2. **Trust:** After your Section 4 evaluation, would you trust this system to run unattended?
   What single evaluation result most influenced your answer?
3. **Cost and scale:** Estimate (from your `response.usage` numbers) the tokens needed to
   process 1,000 applications per month. What does that imply for provider choice?
4. **Looking back at the course:** You have now used classical ML (Lab 2), trained neural
   networks (Lab 3), and used a foundation model via API (Lab 4). For a task like this one,
   why does calling an API beat training your own model — and when would it not?

> Answer:
Prompting as engineering: Prompting uses natural language instructions, while tuning hyperparameters uses math. Both require systematic trial-and-error evaluation.

Trust: No, I would not run it unattended. The risk of edge-case errors requires a human officer to review outputs.

Cost and scale: Processing 1,000 letters (~800k tokens) monthly on an open-weight model via Groq costs under $1 USD, making it highly affordable.

Looking back at the course: An API model is best here because foundation models understand language out of the box. Training a custom model is only necessary if strict privacy mandates require local, offline hardware.

---
### Submission checklist

- [ ] All cells run top-to-bottom with no errors (`Kernel -> Restart & Run All`).
- [ ] **No API key anywhere in the notebook or the commit history.**
- [ ] Every **Student Reasoning** box is filled in with full sentences.
- [ ] `prompts.py` / `prompts.md` committed with your final prompt templates.
- [ ] Evaluation tables and adversarial test outputs visible in the saved notebook.
- [ ] Notebook pushed to `lab-4-llm-decision-support` with incremental commits.
- [ ] Repository link submitted to the course portal.
- [ ] AI Declaration form in Repository.